# NeoOLAF / RAGTree — DocRED Raw-Text Entity + Relation Smoke5 Run + Eval

This notebook does **both** steps for the correct raw-text setting:

1. run prediction on **exactly 5 DocRED documents**;
2. evaluate those same 5 documents.

Important: the extraction prompt uses only raw text + title + global relation vocabulary. It does **not** show DocRED gold/source entity IDs to the model. Gold entities are used only in the evaluation cell.

Metrics produced:

- `entity_inventory`: predicted entity labels/aliases mapped to gold entity clusters.
- `entity_endpoint`: gold entities used as endpoints in relation triples.
- `relation_strict`: strict `(head_gold_id, relation_id, tail_gold_id)` after mapping predicted entities to gold clusters.


In [ ]:
from pathlib import Path
import os, json, subprocess, shlex, time, sys

ROOT = Path("/home/galencarmedeiro/git/postdoc/NeoOLAF")
DATASET_JSONL = Path("/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl")
RUNS_DIR = ROOT / "examples/RAGTreeDatasets/runs"
LOGS_DIR = RUNS_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

TYPE_FILTER = "dev"
MAX_DOCS = 5
DOCUMENT_WORKERS = 2

MODEL_NAME = "openai/gpt-oss-20b"
HOST = "https://openrouter.ai/api"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 600
TEMPERATURE = 0.0

DOCRED_RAW_RETRIES = 3
DOCRED_RAW_RETRY_SLEEP = 2.0
DOCRED_RAW_SCORING_CALIBRATION = True
DOCRED_RAW_DISABLE_HINTS = False
DOCRED_RAW_FOCUS_RELATION_IDS = None
DOCRED_RAW_MAX_RELATIONS = None
TEXT_CHAR_LIMIT = None

PRED_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_predictions.canonical.jsonl"
SUMMARY_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_run_summary.json"
ERROR_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_errors.jsonl"
ARTIFACTS_ROOT = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_artifacts"
VOCAB_PATH = RUNS_DIR / "docred_allowed_relations_raw_er_smoke5.json"
LOG_PATH = LOGS_DIR / "neoolaf_docred_raw_er_smoke5_run.log"
PID_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5.pid"

# Gold slice used ONLY for evaluation. It contains the same first 5 selected docs.
GOLD_SMOKE5_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_gold.jsonl"
EVAL_JSON_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_eval.json"
EVAL_CSV_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_eval_per_doc.csv"

print("ROOT=", ROOT)
print("DATASET_JSONL=", DATASET_JSONL)
print("PRED_PATH=", PRED_PATH)
print("GOLD_SMOKE5_PATH=", GOLD_SMOKE5_PATH)
print("LOG_PATH=", LOG_PATH)
print("OPENROUTER_API_KEY present:", bool(os.environ.get("OPENROUTER_API_KEY")))


## 1. Create the 5-document gold slice for evaluation

This does **not** feed gold entities to the model. It only creates an evaluation file with the same first 5 selected documents that the runner will process.


In [ ]:
assert ROOT.exists(), ROOT
assert DATASET_JSONL.exists(), DATASET_JSONL

selected = []
with DATASET_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)
        if TYPE_FILTER == "all" or rec.get("type") == TYPE_FILTER or rec.get("split") == TYPE_FILTER:
            selected.append(rec)
            if len(selected) >= MAX_DOCS:
                break

assert len(selected) == MAX_DOCS, f"Expected {MAX_DOCS} docs, got {len(selected)}"
with GOLD_SMOKE5_PATH.open("w", encoding="utf-8") as out:
    for rec in selected:
        out.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Wrote", len(selected), "gold/eval docs to", GOLD_SMOKE5_PATH)
for i, rec in enumerate(selected, 1):
    print(i, rec.get("document_id") or rec.get("id") or rec.get("title"), "|", rec.get("title"))


## 2. Run raw-text entity + relation prediction on those 5 documents

This is synchronous: the cell waits until the 5-document run finishes. Do not run evaluation before this cell completes.


In [ ]:
assert os.environ.get("OPENROUTER_API_KEY"), "Set OPENROUTER_API_KEY in the shell before starting Jupyter. Do not put the key in the notebook."
assert (ROOT / "experiments/methods/run_neoolaf_docred_raw_er.py").exists(), "Missing raw ER runner. Apply the raw-text entity+relation patch first."

for p in [PRED_PATH, SUMMARY_PATH, ERROR_PATH, LOG_PATH, EVAL_JSON_PATH, EVAL_CSV_PATH]:
    p.unlink(missing_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(ROOT / "experiments/methods/run_neoolaf_docred_raw_er.py"),
    "--dataset-jsonl-path", str(DATASET_JSONL),
    "--output-jsonl-path", str(PRED_PATH),
    "--summary-output-path", str(SUMMARY_PATH),
    "--error-log-jsonl-path", str(ERROR_PATH),
    "--artifacts-root", str(ARTIFACTS_ROOT),
    "--type-filter", TYPE_FILTER,
    "--max-docs", str(MAX_DOCS),
    "--document-workers", str(DOCUMENT_WORKERS),
    "--backend-name", "openrouter",
    "--host", HOST,
    "--model-name", MODEL_NAME,
    "--temperature", str(TEMPERATURE),
    "--max-tokens", str(MAX_TOKENS),
    "--request-timeout", str(REQUEST_TIMEOUT),
    "--openrouter-reasoning-effort", "minimal",
    "--openrouter-exclude-reasoning",
    "--relation-vocab-dataset-path", str(DATASET_JSONL),
    "--relation-vocab-output-path", str(VOCAB_PATH),
    "--docred-raw-retries", str(DOCRED_RAW_RETRIES),
    "--docred-raw-retry-sleep", str(DOCRED_RAW_RETRY_SLEEP),
]
if DOCRED_RAW_SCORING_CALIBRATION:
    cmd.append("--docred-raw-scoring-calibration")
if DOCRED_RAW_DISABLE_HINTS:
    cmd.append("--docred-raw-disable-hints")
if DOCRED_RAW_FOCUS_RELATION_IDS:
    cmd.extend(["--docred-raw-focus-relation-ids", DOCRED_RAW_FOCUS_RELATION_IDS])
if DOCRED_RAW_MAX_RELATIONS is not None:
    cmd.extend(["--docred-raw-max-relations", str(DOCRED_RAW_MAX_RELATIONS)])
if TEXT_CHAR_LIMIT is not None:
    cmd.extend(["--text-char-limit", str(TEXT_CHAR_LIMIT)])

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))
print("\nRunning...\n")

with LOG_PATH.open("w", encoding="utf-8") as log_f:
    proc = subprocess.Popen(
        cmd,
        cwd=str(ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    PID_PATH.write_text(str(proc.pid), encoding="utf-8")
    for line in proc.stdout:
        print(line, end="")
        log_f.write(line)
        log_f.flush()
    rc = proc.wait()

print("\nReturn code:", rc)
if rc != 0:
    print("Last log lines:")
    print("\n".join(LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
    raise RuntimeError(f"Raw ER smoke5 run failed with return code {rc}")

assert PRED_PATH.exists(), f"Prediction file was not created: {PRED_PATH}"
pred_lines = [x for x in PRED_PATH.read_text(encoding="utf-8").splitlines() if x.strip()]
print("Prediction lines:", len(pred_lines))
assert len(pred_lines) == MAX_DOCS, f"Expected {MAX_DOCS} prediction lines, got {len(pred_lines)}. Check log: {LOG_PATH}"

if SUMMARY_PATH.exists():
    print("Summary:")
    print(SUMMARY_PATH.read_text(encoding="utf-8"))
if ERROR_PATH.exists() and ERROR_PATH.stat().st_size:
    print("Errors present:")
    print(ERROR_PATH.read_text(encoding="utf-8")[:4000])


## 3. Inspect the 5 predictions before evaluation

In [ ]:
for i, line in enumerate(PRED_PATH.open(encoding="utf-8"), 1):
    r = json.loads(line)
    ents = r.get("prediction", {}).get("entities", [])
    rels = r.get("prediction", {}).get("relations", [])
    print("\n" + "=" * 100)
    print(i, r.get("document_id"), "|", r.get("title"), "parsed_ok=", r.get("parsed_ok"))
    print("entities:", len(ents), "relations:", len(rels))
    for e in ents[:20]:
        print(f"  ENT {e.get('entity_id') or e.get('id')}: {e.get('label')} [{e.get('type')}]")
    for rel in rels[:30]:
        print(f"  REL {rel.get('head') or rel.get('head_entity_id')} -- {rel.get('relation_id') or rel.get('relation')} --> {rel.get('tail') or rel.get('tail_entity_id')}")


## 4. Evaluate only the 5-document smoke slice

This evaluates `PRED_PATH` against `GOLD_SMOKE5_PATH`, not the full dataset.


In [ ]:
assert (ROOT / "experiments/methods/evaluate_docred_raw_er.py").exists(), "Missing raw ER evaluator. Apply the raw-text entity+relation patch first."
assert PRED_PATH.exists(), PRED_PATH
assert GOLD_SMOKE5_PATH.exists(), GOLD_SMOKE5_PATH

pred_line_count = sum(1 for _ in PRED_PATH.open(encoding="utf-8"))
gold_line_count = sum(1 for _ in GOLD_SMOKE5_PATH.open(encoding="utf-8"))
print("pred_line_count=", pred_line_count)
print("gold_line_count=", gold_line_count)
assert pred_line_count == MAX_DOCS, f"Expected {MAX_DOCS} predictions, got {pred_line_count}"
assert gold_line_count == MAX_DOCS, f"Expected {MAX_DOCS} gold docs, got {gold_line_count}"

eval_cmd = [
    sys.executable, str(ROOT / "experiments/methods/evaluate_docred_raw_er.py"),
    "--gold-jsonl-path", str(GOLD_SMOKE5_PATH),
    "--pred-jsonl-path", str(PRED_PATH),
    "--type-filter", "all",
    "--output-json-path", str(EVAL_JSON_PATH),
    "--output-csv-path", str(EVAL_CSV_PATH),
]
print(" ".join(shlex.quote(x) for x in eval_cmd))
res = subprocess.run(eval_cmd, cwd=str(ROOT), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Evaluation failed")
print("EVAL_JSON_PATH=", EVAL_JSON_PATH)
print("EVAL_CSV_PATH=", EVAL_CSV_PATH)


## 5. Pretty-print global smoke5 metrics

In [ ]:
metrics = json.loads(EVAL_JSON_PATH.read_text(encoding="utf-8"))
print("docs_gold=", metrics.get("docs_gold"), "docs_pred=", metrics.get("docs_pred"))
for key in ["entity_inventory", "entity_endpoint", "relation_strict"]:
    m = metrics[key]
    print("\n" + key)
    print(f"gold={m['gold']} pred={m['pred']} TP={m['TP']} FP={m['FP']} FN={m['FN']}")
    print(f"precision={m['precision']:.4f} recall={m['recall']:.4f} f1={m['f1']:.4f}")


## 6. Show per-document metrics

In [ ]:
import csv
rows = metrics.get("per_document", [])
for row in rows:
    print("\n" + "=" * 100)
    print(row.get("document_id"), "|", row.get("title"), "parsed_ok=", row.get("parsed_ok"))
    print(f"entity:   TP={row.get('entity_TP')} FP={row.get('entity_FP')} FN={row.get('entity_FN')} F1={row.get('entity_f1'):.4f}")
    print(f"relation: TP={row.get('relation_TP')} FP={row.get('relation_FP')} FN={row.get('relation_FN')} F1={row.get('relation_f1'):.4f}")


## 7. Package smoke5 outputs

In [ ]:
zip_path = ROOT / "neoolaf_docred_raw_er_smoke5_run_eval_results.zip"
files = [
    PRED_PATH,
    SUMMARY_PATH,
    ERROR_PATH,
    GOLD_SMOKE5_PATH,
    VOCAB_PATH,
    EVAL_JSON_PATH,
    EVAL_CSV_PATH,
    LOG_PATH,
    ROOT / "experiments/methods/run_neoolaf_docred_raw_er.py",
    ROOT / "experiments/methods/evaluate_docred_raw_er.py",
    ROOT / "examples/RAGTreeDatasets/RAGTreeDatasets_DocRED_Raw_EntityRelation_Smoke5_RunAndEval_NeoOLAF.ipynb",
]
existing = [str(p.relative_to(ROOT)) for p in files if p.exists()]
cmd = ["zip", "-r", str(zip_path), *existing]
print(" ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)
print(zip_path)
